# LLM Orchestrator — Google Colab

Bootstrap notebook to run the **LLM-Orchestrator** project from GitHub on Google Colab.

The repository contains two independent Poetry projects: `orchestrator` and `provider`.

The setup cell:
- clones the repository (or pulls the latest changes if already present);
- installs Poetry;
- configures Poetry to reuse Colab's Python environment (no separate virtualenv);
- installs the dependencies of **both** projects.


In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/gmonitillodev-hub/LLM-Orchestrator.git"
REPO_DIR = "/content/LLM-Orchestrator"
# Each of these subfolders is its own Poetry project (pyproject.toml), not the repo root.
PROJECT_NAMES = ["orchestrator", "provider"]
PROJECT_DIRS = {name: os.path.join(REPO_DIR, name) for name in PROJECT_NAMES}


def run(command, cwd=None):
    print(f"$ {command}")
    subprocess.run(command, shell=True, check=True, cwd=cwd)


if not os.path.exists(REPO_DIR):
    print("Repository not found: cloning...")
    run(f"git clone {REPO_URL} {REPO_DIR}")
else:
    print("Repository already present: pulling latest changes...")
    run("git pull", cwd=REPO_DIR)

run("python -m pip install -q poetry")

for name, project_dir in PROJECT_DIRS.items():
    print(f"\nInstalling '{name}' ({project_dir})...")
    run("poetry config virtualenvs.create false --local", cwd=project_dir)
    run("poetry install", cwd=project_dir)

    # Make each project's `src` importable directly (e.g. `from classes import ...`)
    # without having to prefix every command with `poetry run`.
    src_dir = os.path.join(project_dir, "src")
    if src_dir not in sys.path:
        sys.path.insert(0, src_dir)

print("\nSetup complete.")


## Environment check


In [ ]:
import os
import sys

print("Python:", sys.version)
for name, project_dir in PROJECT_DIRS.items():
    print(f"\n{name} directory:", project_dir)
    print("\n".join(sorted(os.listdir(project_dir))))


## Update the repository

Run this cell during the same Colab session to fetch the latest changes from GitHub without redoing the full setup.


In [ ]:
run("git pull", cwd=REPO_DIR)


## Playground

From here you can start importing and running the code of both projects.


In [ ]:
# Examples:
# from classes import Request, LlmClient          # orchestrator
# from utils.terminal import get_config            # orchestrator
# from provider.run import run_full                # provider

print("LLM-Orchestrator ready.")


## Run the provider (mock LLM servers)

The `provider` project is a FastAPI server: it must stay running in the background while the orchestrator (or any client) calls it. `run-all` starts the three simulated providers (`Active` on 8000, `Partial` on 8100, `Inactive` on 8200).

This cell starts it as a background subprocess (so the notebook doesn't hang), logs it to `/content/provider.log`, and waits until each endpoint responds to `/health`.


In [ ]:
import subprocess
import time
import urllib.request

PROVIDER_LOG = "/content/provider.log"
PROVIDER_PORTS = {"Active": 8000, "Partial": 8100, "Inactive": 8200}

provider_process = subprocess.Popen(
    "poetry run run-all",
    shell=True,
    cwd=PROJECT_DIRS["provider"],
    stdout=open(PROVIDER_LOG, "w"),
    stderr=subprocess.STDOUT,
)
print(f"Provider starting in background (pid={provider_process.pid}), logs -> {PROVIDER_LOG}")


def wait_for_health(port, timeout=30):
    deadline = time.time() + timeout
    url = f"http://127.0.0.1:{port}/health"
    while time.time() < deadline:
        try:
            urllib.request.urlopen(url, timeout=2)
            return True
        except Exception:
            time.sleep(1)
    return False


for name, port in PROVIDER_PORTS.items():
    is_up = wait_for_health(port)
    print(f"{name} (port {port}):", "up" if is_up else "NOT responding")


## Run the orchestrator

The orchestrator is an interactive CLI: it asks for file paths via `input()`. Run it with the `!` shell magic (not `subprocess.run`) so Colab can forward your input to the process.

Run the cell below to move into the orchestrator's directory, then run the next cell to start it. Answer its prompts directly in the output area.


In [ ]:
import os

os.chdir(PROJECT_DIRS["orchestrator"])
print("Working directory:", os.getcwd())


In [ ]:
!poetry run orchestrator --client Active


## Stop the provider

Run this cell when you're done, to stop the background provider process.


In [ ]:
provider_process.terminate()
provider_process.wait()
print("Provider stopped.")
